# 04 - Workflow Automation

This notebook demonstrates how to automate the expense tracker pipeline using Databricks Workflows.

**Features:**
- Create and manage Databricks Jobs
- Schedule periodic sync
- Set up dependencies between tasks
- Monitor job runs


## Install Required Libraries


In [ ]:
%pip install databricks-sdk --quiet
dbutils.library.restartPython()


## Import Libraries


In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import *
import json


## Initialize Workspace Client


In [ ]:
w = WorkspaceClient()
print(f"✓ Connected to: {w.config.host}")


## Configuration


In [ ]:
# Job configuration
JOB_NAME = "expense-tracker-sync-pipeline"
CLUSTER_SPEC = "shared"  # Use shared cluster for demo

# Get current user for notebook paths
current_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
notebook_base_path = f"/Workspace/Users/{current_user}/expense-tracker/notebooks"

print(f"Job Name: {JOB_NAME}")
print(f"Notebook Path: {notebook_base_path}")


## Create Workflow Definition


In [ ]:
def create_workflow_config():
    """Create workflow configuration for expense tracker"""
    
    workflow_config = {
        "name": JOB_NAME,
        "email_notifications": {
            "on_failure": [current_user]
        },
        "timeout_seconds": 3600,
        "max_concurrent_runs": 1,
        "tasks": [
            {
                "task_key": "sync_expenses",
                "description": "Sync expense data from Lakebase to Lakehouse",
                "notebook_task": {
                    "notebook_path": f"{notebook_base_path}/03-sync-pipeline",
                    "source": "WORKSPACE"
                },
                "existing_cluster_id": "{{CLUSTER_ID}}"  # Replace with actual cluster
            },
            {
                "task_key": "validate_results",
                "description": "Validate synced data",
                "depends_on": [{"task_key": "sync_expenses"}],
                "notebook_task": {
                    "notebook_path": f"{notebook_base_path}/05-validate-results",
                    "source": "WORKSPACE"
                },
                "existing_cluster_id": "{{CLUSTER_ID}}"  # Replace with actual cluster
            }
        ],
        "schedule": {
            "quartz_cron_expression": "0 0 2 * * ?",  # Daily at 2 AM
            "timezone_id": "America/Los_Angeles",
            "pause_status": "PAUSED"  # Start paused for demo
        }
    }
    
    return workflow_config

workflow_config = create_workflow_config()
print("✓ Workflow configuration created")
print(f"\nWorkflow: {workflow_config['name']}")
print(f"Tasks: {len(workflow_config['tasks'])}")
print(f"Schedule: {workflow_config['schedule']['quartz_cron_expression']}")
print(f"\nTasks:")
for task in workflow_config['tasks']:
    print(f"  - {task['task_key']}: {task['description']}")


## Create or Update Job


In [ ]:
def find_job_by_name(job_name):
    """Find existing job by name"""
    try:
        jobs = w.jobs.list(name=job_name)
        for job in jobs:
            if job.settings.name == job_name:
                return job
        return None
    except Exception as e:
        print(f"Error finding job: {str(e)}")
        return None

def create_or_update_job(config):
    try:
        # Check if job exists
        existing_job = find_job_by_name(config['name'])
        
        if existing_job:
            print(f"✓ Job '{config['name']}' already exists (ID: {existing_job.job_id})")
            print(f"  To update, use w.jobs.update(job_id={existing_job.job_id}, ...)")
            return existing_job.job_id
        else:
            print(f"Creating new job: {config['name']}")
            print("Note: Replace {{CLUSTER_ID}} with your actual cluster ID")
            print("\nTo create the job, use:")
            print("  job = w.jobs.create(**workflow_config)")
            return None
            
    except Exception as e:
        print(f"✗ Error: {str(e)}")
        return None

job_id = create_or_update_job(workflow_config)


## Manual Job Trigger Example


In [ ]:
def trigger_job_run(job_id):
    try:
        if job_id:
            print(f"Triggering job run for job_id: {job_id}")
            run = w.jobs.run_now(job_id=job_id)
            print(f"✓ Job run started: {run.run_id}")
            return run.run_id
        else:
            print("No job ID provided - skipping trigger")
            return None
    except Exception as e:
        print(f"✗ Failed to trigger job: {str(e)}")
        return None

# Example: Uncomment to trigger
# run_id = trigger_job_run(job_id)
print("Manual trigger example ready (commented out for safety)")


## Monitor Job Runs

In [ ]:
def list_recent_runs(job_id, limit=5):
    try:
        if not job_id:
            print("No job ID provided")
            return
            
        runs = w.jobs.list_runs(job_id=job_id, limit=limit)
        
        print(f"Recent runs for job {job_id}:")
        print("-" * 80)
        
        for run in runs:
            status = run.state.life_cycle_state
            result = run.state.result_state if hasattr(run.state, 'result_state') else 'N/A'
            start_time = run.start_time if hasattr(run, 'start_time') else 'N/A'
            
            print(f"Run ID: {run.run_id}")
            print(f"  Status: {status}")
            print(f"  Result: {result}")
            print(f"  Started: {start_time}")
            print("-" * 80)
            
    except Exception as e:
        print(f"✗ Failed to list runs: {str(e)}")

# Example usage (when job exists)
# list_recent_runs(job_id)
print("Job monitoring functions ready")


## Alternative: Create Job via UI

For a demo, it's easier to create the job via Databricks UI:


In [ ]:
print("="*80)
print("CREATING JOB VIA UI")
print("="*80)
print("\nSteps:")
print("1. Go to Workflows in the left sidebar")
print("2. Click 'Create Job'")
print("3. Add Task 1: sync_expenses")
print(f"   - Type: Notebook")
print(f"   - Path: {notebook_base_path}/03-sync-pipeline")
print(f"   - Cluster: Select existing cluster")
print("\n4. Add Task 2: validate_results")
print(f"   - Type: Notebook")
print(f"   - Path: {notebook_base_path}/05-validate-results")
print(f"   - Cluster: Select existing cluster")
print(f"   - Depends on: sync_expenses")
print("\n5. Add Schedule (optional):")
print("   - Cron: 0 0 2 * * ? (Daily at 2 AM)")
print("   - Start: Paused")
print("\n6. Save and run manually to test")
print("="*80)


## Export Workflow Configuration

In [ ]:
# Export configuration to file
config_json = json.dumps(workflow_config, indent=2)
print("Workflow Configuration JSON:")
print("="*80)
print(config_json)
print("="*80)
print("\n✓ Configuration can be saved and version controlled")


## Summary


In [ ]:
print("="*80)
print("WORKFLOW AUTOMATION SETUP COMPLETE")
print("="*80)
print("✓ Workflow configuration created")
print("✓ Job management functions defined")
print("✓ Monitoring utilities ready")
print("\nWorkflow Components:")
print("  1. sync_expenses - Sync data from Lakebase to Lakehouse")
print("  2. validate_results - Validate synced data")
print("\nNext Steps:")
print("  1. Create job via UI or SDK")
print("  2. Run notebook 05-validate-results.ipynb")
print("  3. Test end-to-end pipeline")
print("  4. Build Databricks App for user interface")
print("="*80)
